In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Enterprise Retail Data Platform")
    .master("local[*]")
    .config("spark.local.dir", r"F:\PROJECT\spark-temp")
    .config("spark.sql.warehouse.dir", r"F:\PROJECT\spark-warehouse")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [3]:
from pyspark.sql.functions import (
    col,
    count,
    when,
    isnan,
    sum
)

In [4]:
def basic_table_info(df):
    print("="*60)
    print("Rows :", df.count())
    print("column_cnt :", len(df.columns))
    print("column :", df.columns )


In [5]:
def table_schema_info(df):
    print("="*60)
    df.printSchema()


In [6]:
def preview_data(df , n):
    df.show(n, truncate = False)

In [7]:
from pyspark.sql.functions import col, when , sum

def null_check(df):
    df.select([
        sum(
            when(col(c).isNull(),1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    

In [8]:
def duplicate_rows(df):
    total = df.count()
    distinct_cnt = df.distinct().count()
    print("row :", total )
    print("distinct row :", distinct_cnt)
    print("Duplicate rows :", total - distinct_cnt)

In [9]:
DATA_PATH = r"F:\PROJECT\enterprise-retail-data-platform\data\raw"

datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

data_frames = {}

for table_name, file_name in datasets.items():
    data_frames[table_name] = spark.read.csv(
        f"{DATA_PATH}\\{file_name}",
        header=True,
        inferSchema=True
    )

In [10]:
def profile_dataset(df, table_name):
    print("\n" + "=" * 80)
    print(f"DATASET: {table_name.upper()}")
    print("=" * 80)

    print("\n1. Basic Information")
    basic_table_info(df)

    print("\n2. Schema")
    table_schema_info(df)

    print("\n4. Null Value Analysis")
    null_check(df)

    print("\n5. Duplicate Analysis")
    duplicate_rows(df)

    print("\n" + "=" * 80)

In [11]:
for table_name, df in data_frames.items():
    profile_dataset(df, table_name)


DATASET: CUSTOMERS

1. Basic Information
Rows : 99441
column_cnt : 5
column : ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

2. Schema
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


4. Null Value Analysis

5. Duplicate Analysis
row : 99441
distinct row : 99441
Duplicate rows : 0


DATASET: ORDERS

1. Basic Information
Rows : 99441
column_cnt : 8
column : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

2. Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable =